In [1]:
!pip install icecream

In [2]:
!pip uninstall -y scikit-learn

Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2


In [3]:
!pip install scikit-learn==1.5.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 79.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.


In [4]:
!pip install imbalanced-learn

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from icecream import ic
from scipy.stats import skew, kurtosis
from scipy.fft import fft
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

In [6]:

# Set random seed for all libraries
SEED = 42
np.random.seed(SEED)
pd.set_option('mode.chained_assignment', None)  # Disable SettingWithCopyWarning

def aggregate_features(df, has_label=True):
    """
    Extract features from DataFrame and return aggregated DataFrame.
    
    Parameters:
    - df: Input DataFrame (df_train or df_test)
    - has_label: If True, include 'label' column in output (for df_train)
    
    Returns:
    - DataFrame with feature columns and label column (if has_label=True)
    """
    # Group by Sample Name
    grouped = df.groupby('Sample Name')
    
    # Initialize dictionary to store features
    feature_dict = {}
    
    # List of columns for statistical feature extraction
    signal_cols = ['Doin (mV)', 'DOmin (mV)', 'DDO (mV)']
    
    # Iterate through each sample
    for sample_name, group in grouped:
        # Initialize dictionary for current sample
        sample_features = {}
        
        # 1. Basic statistics
        for col in signal_cols:
            data = group[col]
            sample_features[f'mean_{col}'] = data.mean() if not data.empty else 0
            sample_features[f'std_{col}'] = data.std() if len(data) > 1 else 0
            sample_features[f'max_{col}'] = data.max() if not data.empty else 0
            sample_features[f'min_{col}'] = data.min() if not data.empty else 0
            sample_features[f'skew_{col}'] = skew(data) if len(data) > 2 else 0
            sample_features[f'kurtosis_{col}'] = kurtosis(data) if len(data) > 3 else 0
            sample_features[f'q25_{col}'] = data.quantile(0.25) if not data.empty else 0
            sample_features[f'q50_{col}'] = data.quantile(0.50) if not data.empty else 0
            sample_features[f'q75_{col}'] = data.quantile(0.75) if not data.empty else 0
        
        # 2. Time series trend features
        for col in signal_cols:
            data = group[col]
            slopes = (data - data.shift(1)) / (group['No.peak'] - group['No.peak'].shift(1))
            sample_features[f'mean_slope_{col}'] = slopes.mean() if not slopes.empty else 0
            sample_features[f'total_abs_change_{col}'] = np.sum(np.abs(data.diff())) if len(data) > 1 else 0
            sample_features[f'sign_changes_{col}'] = np.sum(np.diff(np.sign(data.diff())) != 0) if len(data) > 1 else 0
        
        # 3. Peak distance features
        peak_diffs = group['No.peak'].diff()
        sample_features['mean_peak_diff'] = peak_diffs.mean() if not peak_diffs.empty else 0
        sample_features['std_peak_diff'] = peak_diffs.std() if len(peak_diffs) > 1 else 0
        
        # 4. Frequency features
        for col in signal_cols:
            signal = group[col].values
            fft_vals = np.abs(fft(signal))
            fft_freq = np.fft.fftfreq(len(signal))
            sample_features[f'dominant_freq_amplitude_{col}'] = np.max(fft_vals) if len(fft_vals) > 0 else 0
            sample_features[f'dominant_freq_{col}'] = np.abs(fft_freq[np.argmax(fft_vals)]) if len(fft_vals) > 0 else 0
            sample_features[f'spectral_power_{col}'] = np.sum(fft_vals ** 2) if len(fft_vals) > 0 else 0
            fft_norm = fft_vals / (np.sum(fft_vals) + 1e-10)
            sample_features[f'spectral_entropy_{col}'] = -np.sum(fft_norm * np.log2(fft_norm + 1e-10)) if len(fft_vals) > 0 else 0
        
        # 5. Variable relationship features
        sample_features['mean_DDO_Doin_ratio'] = (group['DDO (mV)'] / group['Doin (mV)']).mean() if len(group) > 0 else 0
        sample_features['mean_DDO_DOmin_ratio'] = (group['DDO (mV)'] / group['DOmin (mV)']).mean() if len(group) > 0 else 0
        sample_features['std_DDO_Doin_ratio'] = (group['DDO (mV)'] / group['Doin (mV)']).std() if len(group) > 1 else 0
        sample_features['corr_Doin_DOmin'] = group['Doin (mV)'].corr(group['DOmin (mV)']) if len(group) > 1 else 0
        sample_features['corr_Doin_DDO'] = group['Doin (mV)'].corr(group['DDO (mV)']) if len(group) > 1 else 0
        
        # 6. Peak features
        ddo = group['DDO (mV)']
        mean_ddo = ddo.mean() if not ddo.empty else 0
        std_ddo = ddo.std() if len(ddo) > 1 else 0
        sample_features['high_peak_ratio'] = np.sum(ddo > (mean_ddo + std_ddo)) / len(ddo) if len(ddo) > 0 else 0
        max_ddo_peak = group['No.peak'][ddo.idxmax()] if not ddo.empty else 0
        sample_features['mean_dist_to_max_peak'] = np.mean(np.abs(group['No.peak'] - max_ddo_peak)) if len(ddo) > 0 else 0
        close_peaks = np.sum(peak_diffs < peak_diffs.mean()) / (len(peak_diffs) - 1) if len(peak_diffs) > 1 else 0
        sample_features['close_peak_ratio'] = close_peaks
        top_ddo = ddo[ddo > ddo.quantile(0.75)]
        sample_features['std_top_ddo'] = top_ddo.std() if len(top_ddo) > 1 else 0
        
        # 7. Cycle features
        crossings = np.sum(np.diff(np.sign(ddo - mean_ddo)) != 0) / 2
        sample_features['num_cycles'] = crossings if crossings > 0 else 1
        crossing_indices = np.where(np.diff(np.sign(ddo - mean_ddo)) != 0)[0]
        cycle_lengths = np.diff(group['No.peak'].iloc[crossing_indices]) if len(crossing_indices) > 1 else [0]
        sample_features['mean_cycle_length'] = np.mean(cycle_lengths) if len(cycle_lengths) > 0 else 0
        cycle_amplitudes = []
        for i in range(len(crossing_indices) - 1):
            start_idx = crossing_indices[i]
            end_idx = crossing_indices[i + 1]
            cycle_data = ddo.iloc[start_idx:end_idx + 1]
            amplitude = cycle_data.max() - cycle_data.min()
            cycle_amplitudes.append(amplitude)
        mean_amplitude = np.mean(cycle_amplitudes) if len(cycle_amplitudes) > 0 else 0
        std_amplitude = np.std(cycle_amplitudes) if len(cycle_amplitudes) > 0 else 0
        sample_features['mean_cycle_amplitude'] = mean_amplitude
        sample_features['abnormal_cycle_ratio'] = np.sum(np.array(cycle_amplitudes) > (mean_amplitude + std_amplitude)) / len(cycle_amplitudes) if len(cycle_amplitudes) > 0 else 0
        
        # 8. Time interval features
        q1 = peak_diffs.quantile(0.25) if not peak_diffs.empty else 0
        q3 = peak_diffs.quantile(0.75) if not peak_diffs.empty else 0
        iqr = q3 - q1
        sample_features['short_interval_ratio'] = np.sum(peak_diffs < q1) / len(peak_diffs) if not peak_diffs.empty else 0
        sample_features['long_interval_ratio'] = np.sum(peak_diffs > q3) / len(peak_diffs) if not peak_diffs.empty else 0
        sample_features['coeff_variation_intervals'] = (peak_diffs.std() / peak_diffs.mean()) if peak_diffs.mean() != 0 else 0
        outliers = np.sum((peak_diffs < (q1 - 1.5 * iqr)) | (peak_diffs > (q3 + 1.5 * iqr))) / len(peak_diffs) if not peak_diffs.empty else 0
        sample_features['outlier_interval_ratio'] = outliers
        
        # 9. Number of peaks
        sample_features['num_peaks'] = len(group)
        
        # Add label if exists
        if has_label:
            sample_features['label'] = group['label'].iloc[0]
        
        # Append features to dictionary
        for key, value in sample_features.items():
            if key not in feature_dict:
                feature_dict[key] = []
            feature_dict[key].append(value)
    
    # Convert dictionary to DataFrame
    df_aggregated = pd.DataFrame(feature_dict)
    
    # Handle NaN values
    if has_label:
        df_aggregated.fillna({col: 0 for col in df_aggregated.columns if col != 'label'}, inplace=True)
    else:
        df_aggregated.fillna(0, inplace=True)
    
    return df_aggregated

# Read and combine training data
df_gga = pd.read_csv('/kaggle/input/bio-dataset/metadata-gga-2024-10-23.csv')
df_gga['label'] = 'gga' #157
df_gga_metal = pd.read_csv('/kaggle/input/bio-dataset/metadata-gga-metal-2024-10-23.csv')
df_gga_metal['label'] = 'gga-metal' #360 
df_gga_metal_hh = pd.read_csv('/kaggle/input/bio-dataset/metadata-gga-metal-HH-2024-10-23.csv')
df_gga_metal_hh['label'] = 'gga-metal'  #105

# Print sample counts
print("\nSample counts:")
print(f"gga: {len(df_gga['Sample Name'].unique())} samples")
print(f"gga-metal (from df_gga_metal): {len(df_gga_metal['Sample Name'].unique())} samples")
print(f"gga-metal (from df_gga_metal_hh): {len(df_gga_metal_hh['Sample Name'].unique())} samples")

# Combine data
df_train = pd.concat([df_gga, df_gga_metal, df_gga_metal_hh])

# Check input data
for col in ['Doin (mV)', 'DOmin (mV)', 'DDO (mV)', 'No.peak']:
    non_numeric = df_train[col][~pd.to_numeric(df_train[col], errors='coerce').notnull()]
    if not non_numeric.empty:
        print(f"Column {col} contains non-numeric values: {non_numeric.unique()}")

# Create aggregated training dataset
df_train_aggregated = aggregate_features(df_train, has_label=True)

# Select features and labels
features = [col for col in df_train_aggregated.columns if col not in ['label']]
X = df_train_aggregated[features]
y = df_train_aggregated['label']

# Convert column names to string
X.columns = X.columns.astype(str)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)

X_train = X_train.fillna(X_train.median()).replace([float('inf'), -float('inf')], 0)
X_test = X_test.fillna(X_test.median()).replace([float('inf'), -float('inf')], 0)

# Reset indices
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Debug
ic(X_train.head())
ic(X_train.shape)
ic(y_train.head())
ic(len(y_train))

# Encode labels for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Train XGBoost model
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    scale_pos_weight=1,
    random_state=SEED
)
xgb_model.fit(X_train, y_train_encoded)

# Evaluate XGBoost model
xgb_y_pred = le.inverse_transform(xgb_model.predict(X_test))
print("\nXGBoost Model evaluation on test set:")
print(classification_report(y_test, xgb_y_pred))


# Feature importance analysis for XGBoost
xgb_feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
})
xgb_feature_importance = xgb_feature_importance.sort_values('importance', ascending=False)

print("\nTop 20 most important features - XGBoost:")
print(xgb_feature_importance.head(20))

# Plot feature importance for both models
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# XGBoost feature importance
sns.barplot(x='importance', y='feature', data=xgb_feature_importance.head(20), ax=ax2)
ax2.set_title('Top 20 Most Important Features - XGBoost')
ax2.set_xlabel('Importance Score')
ax2.set_ylabel('Feature')

plt.tight_layout()
plt.savefig('feature_importance_comparison.png')
plt.close()

# Inference on test_example.csv
df_test = pd.read_csv('/kaggle/input/test-bio-example/test_example.csv')
df_test_aggregated = aggregate_features(df_test, has_label=False)
df_test_aggregated.fillna(0, inplace=True)

# Make predictions with both models
X_test_final = df_test_aggregated[features]
X_test_final.columns = X_test_final.columns.astype(str)

# XGBoost predictions
xgb_predictions = le.inverse_transform(xgb_model.predict(X_test_final))
xgb_probabilities = xgb_model.predict_proba(X_test_final)

# Print results for all models
print("\nPredictions for samples in test_example.csv:")
for name, xgb_pred, xgb_prob in zip(
    df_test['Sample Name'].unique(),
    xgb_predictions,
    xgb_probabilities,
):
    print(f"\nSample: {name}")
    print(f"XGB Bossting - Prediction: {xgb_pred}, Probability: {xgb_prob}")

# TUNING XGBOOST MODEL
# Define the parameter grid
param_grid = {
         'max_depth': [3, 4],
         'learning_rate': [0.05, 0.1],
         'n_estimators': [100, 150],
         'subsample': [0.6, 0.7],
         'colsample_bytree': [0.8, 0.9],
         'gamma': [0.1, 0.2],
         'reg_lambda': [1.0, 1.5],
         'min_child_weight': [3, 5],
         'scale_pos_weight': [3.0, 4.0]  # Điều chỉnh để cải thiện lớp gga
}

# Apply SMOTE to balance the training data
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_encoded_smote = smote.fit_resample(X_train, y_train_encoded)

# Initialize XGBoost model
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',  # Exact class ratio
    random_state=SEED
)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_weighted',  # Balance precision and recall
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Fit GridSearchCV
grid_search.fit(X_train_smote, y_train_encoded_smote)

# Print best parameters and score
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
xgb_y_pred = le.inverse_transform(best_model.predict(X_test))
print("\nBest XGBoost Model evaluation on test set:")
print(classification_report(y_test, xgb_y_pred))


# Inference on test_example.csv
df_test = pd.read_csv('/kaggle/input/test-bio-example/test_example.csv')
df_test_aggregated = aggregate_features(df_test, has_label=False)
df_test_aggregated.fillna(0, inplace=True)

# Make predictions with best model
X_test_final = df_test_aggregated[features]
X_test_final.columns = X_test_final.columns.astype(str)
xgb_predictions = le.inverse_transform(best_model.predict(X_test_final))
xgb_probabilities = best_model.predict_proba(X_test_final)

# Print results
print("\nPredictions for samples in test_example.csv:")
for name, xgb_pred, xgb_prob in zip(
    df_test['Sample Name'].unique(),
    xgb_predictions,
    xgb_probabilities,
):
    print(f"\nSample: {name}")
    print(f"XGBoost - Prediction: {xgb_pred}, Probability: {xgb_prob}")


Sample counts:
gga: 156 samples
gga-metal (from df_gga_metal): 359 samples
gga-metal (from df_gga_metal_hh): 103 samples


/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sign
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sign
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/


XGBoost Model evaluation on test set:
              precision    recall  f1-score   support

         gga       0.65      0.59      0.62        63
   gga-metal       0.86      0.89      0.88       185

    accuracy                           0.81       248
   macro avg       0.76      0.74      0.75       248
weighted avg       0.81      0.81      0.81       248


Top 20 most important features - XGBoost:
                             feature  importance
28        total_abs_change_Doin (mV)    0.042890
23                 kurtosis_DDO (mV)    0.037218
22                     skew_DDO (mV)    0.034107
10                    std_DOmin (mV)    0.033681
50               mean_DDO_Doin_ratio    0.033591
20                      max_DDO (mV)    0.031805
34         total_abs_change_DDO (mV)    0.030654
1                      std_Doin (mV)    0.029357
19                      std_DDO (mV)    0.028763
46  dominant_freq_amplitude_DDO (mV)    0.025740
18                     mean_DDO (mV)    0.024422
52 

/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sign
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sign
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/

Best parameters found:  {'colsample_bytree': 0.9, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 100, 'reg_lambda': 1.0, 'scale_pos_weight': 3.0, 'subsample': 0.6}
Best cross-validation score:  0.8915448729536763

Best XGBoost Model evaluation on test set:
              precision    recall  f1-score   support

         gga       0.57      0.60      0.58        63
   gga-metal       0.86      0.84      0.85       185

    accuracy                           0.78       248
   macro avg       0.71      0.72      0.72       248
weighted avg       0.79      0.78      0.78       248


Predictions for samples in test_example.csv:

Sample: 27032024-BOD-15-10-Q=49.66mL/phút-1
XGBoost - Prediction: gga, Probability: [0.8774839  0.12251607]

Sample: 27032024-BOD-10-5-Q=50.45mL/phút-1
XGBoost - Prediction: gga, Probability: [0.8637649  0.13623515]

Sample: 29032024-BOD-10-5-Q=50.81mL/phút-2
XGBoost - Prediction: gga, Probability: [0.72626793 0.2737321 ]



/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sign
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sign
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/